In [1]:
# Imports

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
# Selecting model

MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [3]:
# Creating embeddings and vector database

embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory = DB_NAME, embedding_function=embeddings)

In [4]:
# Creating retriever and llm objects

retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature = 0, model_name = MODEL)

In [5]:
# Testing retriever

retriever.invoke("Who is Avery?")

[Document(id='0bc64c2e-fb56-4338-a7f3-50d8eafc28d8', metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [7]:
def answer_question(question:str,history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context = context)
    response = llm.invoke([SystemMessage(content = system_prompt),HumanMessage(content=question)])
    return response.content

In [8]:
# Testing chat function

answer_question("Who is avery",[])

'Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been with the company since its founding in 2015 and has played a key role in establishing Insurellm as a leading player in the insurance technology industry. Avery is known for her innovative leadership, risk management expertise, and her active involvement in professional development, diversity initiatives, and community outreach.'

In [9]:
# Launching gradion interface

gr.ChatInterface(answer_question).launch()

c:\Users\Riad\Desktop\Udemy Courses\Udemy AI Engineer Core Track\MyTry\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
